![DB Academy](https://files.training.databricks.com/binder/prod_main/automated-deployment-with-declarative-automation-bundles-en_us-2.4.0/images/20260828T161450Z/Automated Deployment with Declarative Automation Bundles/Includes/images/common/db-academy.png)

# 08 - Deploying a Declarative Automation Bundle (DAB) to Multiple Environments

## Overview

In this demonstration, you'll deploy the same job to two environments (`development` and `production`) from a single bundle. You'll work with **bundle variables**, **lookup variables**, the **`include`** mapping for modularizing resource files, and the **`targets`** mapping to override per-environment settings (catalog, compute, mode).

The example builds on the simple bundle from the previous demonstration. The key new ideas here are: defining variables once and reusing them, splitting resources into their own YAML files, and overriding values at the target level so dev and prod can differ without duplicating job definitions.

## Learning Objectives

By the end of this demonstration, you will be able to:

1. **Define and reference bundle variables** in `databricks.yml`, including a **lookup** variable that resolves a cluster name to a cluster ID.
2. **Modularize resources** by moving a job definition into its own YAML file under `resources/` and referencing it via the `include` mapping.
3. **Override values per target** by setting `mode`, `variables`, and task-level overrides differently for `development` vs `production`.
4. **Inspect a fully-resolved bundle** with `databricks bundle validate --output json` to see what will actually be deployed.
5. **Deploy, run, and destroy** the same bundle against two targets using `databricks bundle deploy -t <target>`, `databricks bundle run -t <target> <job_key>`, and `databricks bundle destroy --auto-approve`.

## REQUIRED - SELECT A COMPUTE ENVIRONMENT
<div style="
  border-left: 4px solid #f44336;
  background: #ffebee;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#c62828; margin-bottom:6px; font-size: 1.1em;">Select All-Purpose Compute</strong>
  <div style="color:#333;">

This notebook requires **all-purpose compute** (Dedicated). Serverless is not supported for this notebook.

Follow these steps to attach an all-purpose compute cluster:

1. Navigate to the top-right of this notebook and click the drop-down menu to select your `labuser_USERNAME` cluster.
    - By default, the notebook might use **Serverless**.

2. If your cluster is available, select it and continue to the next cell. If the cluster is not shown:

    - In the drop-down, select **More**.

    - In the **Attach to an existing compute resource** pop-up, select the first drop-down. You will see a unique cluster name in that drop-down. Please select that cluster.

⚠️ **NOTE:** If the cluster shows a **terminated** state (red dot in the cluster picker), it needs to be started before you can attach. Click the cluster, then **Start**, and wait a few minutes until you see a green dot.
  </div>
</div>



## REQUIRED - DATA SETUP

<div style="
  border-left: 4px solid #f44336;
  background: #ffebee;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#c62828; margin-bottom:6px; font-size: 1.1em;">Data Setup</strong>
  <div style="color:#333;">

Recall that your environment was setup using the **02 - REQUIRED - Course Setup and Authentication**.

If you end your lab or your lab session times out, your environment will be reset. You will need to rerun the **02 - REQUIRED - Course Setup and Authentication** notebook to recreate the catalogs and data for your environment.

  </div>
</div>




## A. Classroom Setup

Run the following cell to configure your working environment for this course.

In [0]:
%run ../Includes/Classroom-Setup-08

1. Run the Databricks CLI command below to confirm the Databricks CLI is authenticated.

In [0]:
%sh
databricks catalogs list

<div style="
  border-left: 4px solid #ff9800;
  background: #fff3e0;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#e65100; margin-bottom:6px; font-size: 1.1em;">
    DATABRICKS CLI ERROR TROUBLESHOOTING:
  </strong>
  <div style="color:#333;">

  - If you encounter a Databricks CLI authentication error, it means the authentication was not successful. Confirm you ran the notebook using your **all purpose compute**.

  - If you encounter the error below, it means your **databricks.yml** file has syntax issues due to a modification. Even for non-DAB CLI commands, the **databricks.yml** file is still required, as it may contain important authentication details, such as the host and profile, which are utilized by the CLI commands.

![CLI Invalid YAML](https://files.training.databricks.com/binder/prod_main/automated-deployment-with-declarative-automation-bundles-en_us-2.4.0/images/20260828T161450Z/Automated Deployment with Declarative Automation Bundles/Includes/images/databricks_cli_error_invalid_yaml.png)
  </div>
</div>


2. Run the `databricks -v` command to view the version of the Databricks CLI. 

    Confirm that the cell returns version **v0.298.0**.

In [0]:
%sh
databricks -v

## B. Explore the Development and Production Data

1. Preview the development data in your **labuser_UNIQUE_ID_1_dev** catalog. Note the following:
   - It contains 7,500 rows (excluding the header column).
   - The PII data is masked.

   **NOTE:** In this scenario, the sample data in the development environment is a subset of production data used for testing. We will test against this dataset later.

In [0]:
spark.sql(f'''
SELECT *
FROM text.`/Volumes/{catalog_dev}/default/health`
''').display()

2. Preview the production data in your **labuser_UNIQUE_ID_3_prod** catalog. Note the following:
   - It contains 70,695 rows.
   - The PII data is available.

   **NOTE:** In our scenario, a CSV file is added to the **health** volume in the prod catalog daily. If you inspect the **health** volume in the production catalog, you will find 3 days already populated.

In [0]:
spark.sql(f'''
SELECT count(*) AS Total
FROM text.`/Volumes/{catalog_prod}/default/health`
''').display()

In [0]:
spark.sql(f'''
SELECT *
FROM text.`/Volumes/{catalog_prod}/default/health`
''').display()

## C. Deploy a DAB to Multiple Environments (Development and Production)

In this example, we will be using the same job from the **05 - Deploying a Simple DAB** demonstration. Here are the desired configurations of each environment (catalog):

#### Development target configuration requirements:
- Use the value **labuser_UNIQUE_ID_1_dev** for the development catalog to read and write to.
- Run the job using the **small lab cluster** since the development data is small and static.
- Make the development environment the **default** environment.

#### Production target configuration requirements:
- Use the value **labuser_UNIQUE_ID_3_prod** for the production catalog to access the production data.
- Run the job using serverless compute since the data will continually grow, letting Databricks Serverless adjust to the compute needs.


**Deployment modes (`development` / `production`)**: [AWS](https://docs.databricks.com/aws/en/dev-tools/bundles/deployment-modes) | [Azure](https://learn.microsoft.com/en-us/azure/databricks/dev-tools/bundles/deployment-modes) | [GCP](https://docs.databricks.com/gcp/en/dev-tools/bundles/deployment-modes)


### C1. Explore the `resources` Job YAML File
1. Open the **./resources/demo_08_job.job.yml** file in a new tab and explore the job configuration.

   a. The job key name is `demo_08_job`.

   b. The job name template is `${bundle.target}_demo08_dab_${workspace.current_user.userName}`. Substitution at deploy time gives you a name that includes the target environment and user.

   c. The job uses the `../src/create_bronze_table.ipynb` and `../src/create_silver_table.ipynb` notebooks.
   
   d. In the YAML file, scroll down and notice that the parameters for this job use variables:
   ```yaml
      parameters:
      - name: display_target
        default: ${bundle.target}
      - name: catalog_name
        default: ${var.target_catalog}
    ```

   e. No cluster is specified, so this job runs on Serverless compute by default.
   
   f. Leave this tab open.

2. Run the following code to obtain your lab user name. You will need this for the next section.

In [0]:
print(my_catalog)

### C2. Explore and Modify the `databricks.yml` File

1. In your other tab, navigate to the **./databricks.yml** file located in the main demonstration folder and explore the bundle. Notice the following:

   a. This bundle is named `demo_08_bundle`.

   b. This bundle contains an `include` top-level mapping:
    - This specifies the path to the **./resources/demo_08_job.job.yml** file.
    - That YAML file defines the job that will be merged into the `resources` mapping at deploy time, as you saw in the previous step.
    - **NOTE:** As your project grows, it's a best practice to modularize the resources of the DAB into per-resource YAML files.

   c. This DAB also contains a `variables` top-level mapping. Let's review the defined variables:

      - The `my_lab_user_name` variable uses the substitution `${workspace.current_user.short_name}`. This obtains your username for the lab and propagates the correct value to the remaining variables for each catalog.

      - The `catalog_dev` variable uses the `my_lab_user_name` variable and appends `_1_dev` to your user name to reference your development catalog.

      - The `catalog_prod` variable uses the `my_lab_user_name` variable and appends `_3_prod` to your user name to reference your production catalog.

      - The `target_catalog` variable defaults to your dev catalog.
        - This variable is referenced in the job parameters defined in **./resources/demo_08_job.job.yml**:

      </br>
    
    ```yaml
      parameters:
      - name: display_target
        default: ${bundle.target}
      - name: catalog_name
        default: ${var.target_catalog}
    ```
   </br>
   
      - The `raw_data_path` variable references the **health** volume using `target_catalog`, which by default points at the **health** volume in your dev catalog.

**Variables and substitutions (`${var.…}`, `${bundle.…}`, lookups)**: [AWS](https://docs.databricks.com/aws/en/dev-tools/bundles/variables) | [Azure](https://learn.microsoft.com/en-us/azure/databricks/dev-tools/bundles/variables) | [GCP](https://docs.databricks.com/gcp/en/dev-tools/bundles/variables)

<div style="
  border-left: 4px solid #ff9800;
  background: #fff3e0;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#e65100; margin-bottom:6px; font-size: 1.1em;">
    TO DO - Set the cluster lookup in <strong>databricks.yml</strong>
  </strong>
  <div style="color:#333;">

The `cluster_id` variable uses a **lookup** to resolve a cluster name into a cluster ID at deploy time. Find the variable in **databricks.yml** and update the `cluster:` value with **your** lab cluster's name.

```yaml
variables:
  cluster_id:
    description: Look up your lab cluster's ID by name.
    lookup:
      cluster: <your-cluster-name>     # <-- paste your cluster name here
```

In the Databricks Academy lab, your cluster name matches the value printed by the cell above (your lab username).

  </div>
</div>

Leave the tab with your **databricks.yml** file open.

2. In the **databricks.yml** file, explore the `targets` first-level mapping. Notice the following:

   a. When deploying to the `development` target:
   
      - It is set to `mode: development`.
      
      - It is the **default** target.
      
      - The `root_path` where files are placed ends with the target name, `development`.
      
      - Compute is overridden for each task in the `resources` mapping. The lookup variable `my_cluster_id` (defined earlier) supplies the small lab cluster's ID. We do this because development data is small and doesn't need large compute.

   b. When deploying to the `production` target:
   
      - It is set to `mode: production`.
      
      - The `target_catalog` variable is overridden from the default `${var.catalog_dev}` to `${var.catalog_prod}`. This makes the deployed job read from and write to the production catalog.
      
      - The `root_path` where files are placed ends with the target name, `production`.

      - The job runs on **Serverless** because we are not overriding the compute defined in the resource YAML.


<div style="
  border-left: 4px solid #1976d2;
  background: #e3f2fd;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#0d47a1; margin-bottom:6px; font-size: 1.1em;">
    Information
  </strong>
  <div style="color:#333;">

- If available, you could specify the `host` and choose which Databricks workspace to deploy to. In this lab we have only one workspace, so we isolate environments by **catalog**.
  
- This example overrides only a few configurations for the `production` target. Many other settings can be overridden, see the **Bundle settings** documentation:
[AWS](https://docs.databricks.com/aws/en/dev-tools/bundles/settings) |
[Azure](https://learn.microsoft.com/en-us/azure/databricks/dev-tools/bundles/settings) |
[GCP](https://docs.databricks.com/gcp/en/dev-tools/bundles/settings)

  </div>
</div>

3. Validate the bundle for this demonstration and confirm it validates correctly.

    **NOTE:** If the bundle does not validate, read the error and fix the issue. Common causes:

    - Missing file extensions on the notebook paths within the **demo_08_job.job.yml** file.

    - Variables not defined or referenced correctly in the **databricks.yml** file.

In [0]:
%sh
databricks bundle validate

<div style="
  border-left: 4px solid #ff9800;
  background: #fff3e0;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#e65100; margin-bottom:6px; font-size: 1.1em;">
     Troubleshooting
  </strong>
  <div style="color:#333;">
If you see the following error after validating your bundle, the format of your notebook could be incorrect.

`Error: notebook src/create_bronze_table.ipynb not found`. 

Check the format of your notebook and adjust accordingly. 

  </div>
</div>



4. You can run `databricks bundle validate --output json` to view the fully-resolved bundle configuration in JSON. This is useful for confirming variable substitutions resolved as expected.

Some commonly used substitutions:

- `${bundle.name}`

- `${bundle.target}`  (preferred over the deprecated `${bundle.environment}`)

- `${workspace.host}`

- `${workspace.current_user.short_name}`

- `${workspace.current_user.userName}`

- `${workspace.file_path}`

- `${workspace.root_path}`

- `${resources.jobs.<job-name>.id}`

- `${resources.models.<model-name>.name}`

- `${resources.pipelines.<pipeline-name>.name}`

For example, in the JSON output below:

- `bundle.target` resolves to `development`. The YAML uses `${bundle.target}` to reference this.
- Look for `workspace` > `current_user` > `short_name`. This returns your lab user name.

In [0]:
%sh
databricks bundle validate --output json

### C3. Deploy to the Development Environment

1. Delete the tables **health_bronze_demo_08** and **health_silver_demo_08** if they exist in our development catalog so we can verify that our bundle creates them on deploy.

    Run the code and confirm the tables are not in your **_1_dev** catalog. The output below will show any tables that exist in the **default** schema other than **health_bronze_demo_08** and **health_silver_demo_08**.

In [0]:
del_table(catalog_dev, 'default', 'health_bronze_demo_08')
del_table(catalog_dev, 'default', 'health_silver_demo_08')

spark.sql(f'''SHOW TABLES IN {catalog_dev}.default''').display()

2. Let's deploy the bundle to the **development** environment using the specific configurations.

    **NOTE:** If you do not specify `-t development`, it will deploy to this environment by default since the configuration `default: True` is used for the development target in the **databricks.yml** file. However, it's better to be explicit.

In [0]:
%sh
databricks bundle deploy -t development

3. When the cell above completes (in about a minute), view the deployed job named `[dev username] development_demo08_dab_<username>`.

    In the job, check the following:

    - Select the job tasks and confirm each task uses the lab compute cluster specified in the configuration.

    - Find the **Job parameters** section in the right details pane. Note the values:

        **Job parameters**
        - `catalog_name` - your `labuser_UNIQUE_ID_1_dev` catalog
        - `display_target` - `development`

    Recall that we deployed the job in `development` mode, and it uses the default variable values defined in **databricks.yml** to read from and write to your development catalog.

#### Checkpoint
![Dev](https://files.training.databricks.com/binder/prod_main/automated-deployment-with-declarative-automation-bundles-en_us-2.4.0/images/20260828T161450Z/Automated Deployment with Declarative Automation Bundles/Includes/images/multiple-env-demo/dev-deployment.png)

4. Run the job in the development environment.

    **NOTE:** While the job is running, let's take a moment to address any specific questions.


In [0]:
%sh
databricks bundle run -t development demo_08_job


<div style="
  border-left: 4px solid #1976d2;
  background: #e3f2fd;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#0d47a1; margin-bottom:6px; font-size: 1.1em;">
    Running Using the Job Key
  </strong>
  <div style="color:#333;">

When running a job from the command line, you will need to pass the job key from the job YAML file. 

For example, in our scenario, we have the following in our job YAML file:


```YAML
  resources:
    jobs:
      demo_08_job:    #<---- Job key
        name: ${bundle.target}_demo08_dab_${workspace.current_user.userName}
        ...
```

So, we will run `databricks bundle run -t development demo_08_job`.

  </div>
</div>



5. Run the following cell to view the available tables in the development catalog after the job completes (in about 2 minutes).

    Notice that the two new tables were created:

    - **health_bronze_demo_08**
    - **health_silver_demo_08**

In [0]:
spark.sql(f'SHOW TABLES IN {catalog_dev}.default').display()

6. Count the number of rows in the **health_bronze_demo_08** table in the **user_name_1_dev** catalog. 

    Notice that it contains 7,500 rows, as we are using the development data.

    This confirms that our job correctly read from and wrote to the **development** catalog.

In [0]:
spark.sql(f'''
    SELECT count(*) 
    FROM {catalog_dev}.default.health_bronze_demo_08''').display()

### C4. Deploy to the Production Environment
Now that we've confirmed the job ran in the development environment, let's deploy the same job to the production environment.

**NOTE:** In real production, you typically run the job using a service principal. 
  - See the **Set a bundle run identity** documentation:
[AWS](https://docs.databricks.com/aws/en/dev-tools/bundles/run-as) |
[Azure](https://learn.microsoft.com/en-us/azure/databricks/dev-tools/bundles/run-as) |
[GCP](https://docs.databricks.com/gcp/en/dev-tools/bundles/run-as)

For demonstration purposes, we are simply running the production job as the user.

1. Before we deploy to production let's check (and delete if necessary) the tables in the `<username>_3_prod` catalog. 

    Notice that the following tables are not present in the production catalog:
      - **health_bronze_demo_08**
      - **health_silver_demo_08**

In [0]:
del_table(catalog_prod, 'default', 'health_bronze_demo_08')
del_table(catalog_prod, 'default', 'health_silver_demo_08')

spark.sql(f'SHOW TABLES IN {catalog_prod}.default').display()

2. Let's view the **production** configurations. 

    Note the following:

```YAML
  production:
    mode: production
    workspace:
      # host: https://dbc-d9be2316-40bd.cloud.databricks.com/
      root_path: /Workspace/Users/${workspace.current_user.userName}/.bundle/${bundle.name}/${bundle.target}

    ## Change variable values when in the production environment to use the production catalog username_3_prod
    variables:
        target_catalog: ${var.catalog_prod}
```


<div style="
  border-left: 4px solid #1976d2;
  background: #e3f2fd;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#0d47a1; margin-bottom:6px; font-size: 1.1em;">
    Information
  </strong>
  <div style="color:#333;">

- Here, we are modifying the variable `target_catalog` to reference our variable `catalog_prod` that references our production catalog. This overrides the default job parameter in the **./resources/demo_08_job.job.yml** file.

- We are not adding any overrides regarding the cluster to use for our job. Since we do not provide any overrides, it will use the default set in the **./resources/demo_08_job.job.yml** file, which is using Serverless compute.

  </div>
</div>



3. Let's deploy the bundle to the **production** environment using the specified configurations.

In [0]:
%sh
databricks bundle deploy -t production

4. When the cell above completes (in about a minute), view the deployed job named `production_demo08_dab_<username>`.

    In the job, check the following:

    - Select the job tasks and confirm each task uses Serverless compute.

    - Find the **Job parameters** section in the right details pane. Note the values:

        **Job parameters**
        - `catalog_name` - your `labuser_UNIQUE_ID_3_prod` catalog
        - `display_target` - `production`

    Recall that we deployed the job in `production` mode, and it uses the configuration we specified in **databricks.yml** to read from and write to the production catalog and to use Serverless compute.

#### Checkpoint
![Dev](https://files.training.databricks.com/binder/prod_main/automated-deployment-with-declarative-automation-bundles-en_us-2.4.0/images/20260828T161450Z/Automated Deployment with Declarative Automation Bundles/Includes/images/multiple-env-demo/prod-deployment.png)


5. Run the production job using the Databricks CLI.

In [0]:
%sh
databricks bundle run -t production demo_08_job

6. While the job is running, let's view where the Databricks assets were bundled.

    a. In the main navigation bar, right-click on **Workspace** and select *Open in a New Tab*.

    b. Navigate to **Workspace > Users > your user name**.

    c. Open the **.bundle** folder. Here, you should see the names of the bundles you have deployed (**demo05_bundle** and **demo_08_bundle**).

    d. Open the deployed **demo_08_bundle** (the bundle name we specified in the **databricks.yml** file for this demonstration).

    e. Here, we can see that we deployed to the **development** and **production** targets. 

    f. Select the **production** folder.
    - You will see the **artifacts**, **files**, and **state** folders.

    g. Select the **files** folder.
    - Notice that all of the files we deployed have been added to this location for the production mode deployment within the Workspace.

    h. Close this tab.

7. By now, the **production** job should be completed. 

    Navigate to the job and confirm it executed successfully.

8. Run the following code to view the tables in your `labuser_UNIQUE_ID_3_prod` catalog. 

    Notice that the production job created the production tables:
      - **health_bronze_demo_08**
      - **health_silver_demo_08**

In [0]:
spark.sql(f'SHOW TABLES IN {catalog_prod}.default').display()

9. Count the number of rows in the **health_bronze_demo_08** table in the **labuser_UNIQUE_ID_3_prod** catalog. 

  Notice that it contains over 70,692 rows because it's read from the production data and writing to the production catalog.

In [0]:
spark.sql(f'''
          SELECT count(*) 
          FROM {catalog_prod}.default.health_bronze_demo_08'''
          ).display()

## D. Destroy the Bundles
Lastly, since we are finished with this bundle, let's delete it using the `databricks bundle destroy` command.


  By default, you are prompted to confirm permanent deletion of the previously-deployed jobs, pipelines, and artifacts. To skip these prompts and perform automatic permanent deletion, add the `--auto-approve` option to the bundle destroy command.

1. Delete the bundles!

In [0]:
%sh
databricks bundle destroy --auto-approve
databricks bundle destroy -t production --auto-approve


<div style="
  border-left: 4px solid #f44336;
  background: #ffebee;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#c62828; margin-bottom:6px; font-size: 1.1em;">Warning!</strong>
  <div style="color:#333;">

Destroying a bundle permanently deletes a bundle's previously-deployed jobs, pipelines, and artifacts. This action cannot be undone.

For more information, view the **Destroy the bundle** documentation:
[AWS](https://docs.databricks.com/aws/en/dev-tools/bundles/work-tasks#step-6-destroy-the-bundle) |
[Azure](https://learn.microsoft.com/en-us/azure/databricks/dev-tools/bundles/work-tasks#step-6-destroy-the-bundle) |
[GCP](https://docs.databricks.com/gcp/en/dev-tools/bundles/work-tasks#step-6-destroy-the-bundle)

  </div>
</div>

## Conclusion

In this demonstration, you deployed the same bundle to two targets and saw how to keep dev and prod from drifting apart:

1. Modularized the job by moving its definition into **./resources/demo_08_job.job.yml** and pulling it in via the `include` mapping.
2. Defined reusable **variables** (`my_lab_user_name`, `catalog_dev`, `catalog_prod`, `target_catalog`, `raw_data_path`) and a **lookup** variable (`my_cluster_id`) that resolves a cluster name to a cluster ID.
3. Used `databricks bundle validate --output json` to inspect the fully-resolved bundle and confirm substitutions.
4. Deployed and ran the bundle against both `development` and `production` targets, with each target overriding the catalog and compute as needed.
5. Cleaned up by destroying both targets with `databricks bundle destroy --auto-approve` and `databricks bundle destroy -t production --auto-approve`.

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>